# R2-10 · Series temporales — Profundización (opcional) 🔬

**Formación Pública — Rama R2 · Científico de Datos · Notebook de profundización**

Opcional y de **transferencia**: las mismas herramientas sobre otra serie del país — los
**sismos** de magnitud 4,5 o más registrados por el USGS en Chile entre 2020 y 2025. Primero hay
que **convertir eventos en serie** (conteo mensual), y después responder con honestidad una
pregunta que suena razonable: *¿hay meses más sísmicos que otros?*

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import os, urllib.request
import numpy as np, pandas as pd
CSV = "sismos.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{CSV}", CSV)
sismos = pd.read_csv(CSV, parse_dates=["fecha"])   # USGS: sismos M≥4,5 en Chile, 2020–2025
print("sismos:", sismos.shape, sismos["fecha"].min().date(), "→", sismos["fecha"].max().date())
sismos.head(3)

## 1. De eventos a serie: conteo mensual

Un catálogo de sismos no es una serie temporal: es una lista de **eventos**. Se convierte en serie **contando** eventos por período. `resample("MS").size()` cuenta por mes (y un mes sin sismos aparecería con 0, no desaparecería).

Si los sismos ocurrieran al azar e independientes (proceso de Poisson), la **varianza** del conteo sería parecida a su **media**. Si la varianza es bastante mayor, los eventos vienen **en racimos**.

### ✍️ Ejercicio 1 — Conteo mensual, media y razón varianza/media

In [ ]:
conteo = sismos.set_index("fecha").resample("MS").size()
media_mensual = conteo.mean()
razon_var_media = conteo.var() / conteo.mean()
print("meses:", len(conteo), f"| media: {media_mensual:.1f} sismos/mes | varianza/media: {razon_var_media:.2f}")

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    _c = sismos.set_index("fecha").resample("MS").size()
    assert len(conteo) == len(_c) == 72 and int(conteo.sum()) == len(sismos)
    assert abs(razon_var_media - _c.var() / _c.mean()) < 1e-9
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'sismos.set_index("fecha").resample("MS").size(); conteo.var() / conteo.mean().')
    print("   Detalle:", e)

## 2. El perfil por mes del calendario (y quién lo mueve)

Igual que con el PM2.5: promedia el conteo por **mes del calendario**. Después busca el **día** con más sismos del catálogo: ¿cae en el mes «más sísmico»?

### ✍️ Ejercicio 2 — Perfil mensual y día récord

In [ ]:
perfil = conteo.groupby(conteo.index.month).mean()
mes_max = int(perfil.idxmax())
dia_record = sismos["fecha"].value_counts().idxmax()
print(perfil.round(1).to_dict())
print("Mes con más sismos en promedio:", mes_max)
print("Día récord:", dia_record.date(), "con", sismos["fecha"].value_counts().max(), "sismos")
print(sismos[sismos["fecha"] == dia_record].nlargest(3, "magnitud")[["magnitud", "lugar"]])

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert mes_max == int(conteo.groupby(conteo.index.month).mean().idxmax())
    assert pd.Timestamp(dia_record) == sismos["fecha"].value_counts().idxmax()
    assert pd.Timestamp(dia_record).month == mes_max, "¿el día récord cae en el mes máximo?"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'conteo.groupby(conteo.index.month).mean(); .idxmax(); sismos["fecha"].value_counts().idxmax().')
    print("   Detalle:", e)

## 3. ¿Estacionalidad real o racimos? Un test honesto

La celda de abajo aplica un test **chi-cuadrado** a los sismos individuales por mes del calendario: da un p-valor bajo, que parece decir «hay estacionalidad». **Pero ese test supone eventos independientes**, y las réplicas no lo son: un solo terremoto aporta decenas de sismos al mismo mes.

Un test que respeta los racimos: **permutar meses completos**. Si el calendario no importa, barajar los 72 conteos mensuales entre sí debería producir perfiles tan dispares como el real. El p-valor es la fracción de barajadas con una diferencia máx−mín **igual o mayor** que la real.

In [ ]:
# El test ingenuo (ilustrativo): chi-cuadrado sobre sismos individuales por mes del calendario,
# con lo esperado proporcional a los días de cada mes. Supone eventos INDEPENDIENTES.
from scipy.stats import chisquare
obs = sismos["fecha"].dt.month.value_counts().sort_index()
dias = pd.Series(pd.date_range("2020-01-01", "2025-12-31").month).value_counts().sort_index()
esperado = obs.sum() * dias / dias.sum()
print(f"chi-cuadrado ingenuo: p = {chisquare(obs.values, esperado.values).pvalue:.3f}  ← parece 'significativo'")

### ✍️ Ejercicio 3 — Test de permutación

In [ ]:
observado = perfil.max() - perfil.min()
rng = np.random.default_rng(42)
mayores = 0
N = 2000
for _ in range(N):
    barajado = pd.Series(rng.permutation(conteo.values), index=conteo.index)
    p = barajado.groupby(barajado.index.month).mean()
    mayores += (p.max() - p.min()) >= observado
p_valor = mayores / N
print(f"diferencia observada: {observado:.1f} sismos/mes · p-valor por permutación: {p_valor:.2f}")

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert 0.10 < p_valor < 0.60, "con 2000 permutaciones el p-valor debería rondar 0,3"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'p = barajado.groupby(barajado.index.month).mean(); mayores += (p.max() - p.min()) >= observado.')
    print("   Detalle:", e)

## 4. Entonces, ¿cómo se pronostica?

Valida igual que con el PM2.5: entrena con 2020–2024 y pronostica los 12 meses de 2025 con (a) el **promedio** plano de todos los meses, (b) el **promedio estacional** y (c) el **ingenuo** (último mes). Si la estacionalidad fuera real, (b) debería ganar con claridad.

### ✍️ Ejercicio 4 — Tres pronósticos de conteo para 2025

In [ ]:
entrena = conteo[:"2024"]
real = conteo["2025"]
errores = {
    "promedio": float(np.abs(entrena.mean() - real.values).mean()),
    "promedio_estacional": float(np.abs(entrena.groupby(entrena.index.month).mean().values - real.values).mean()),
    "ingenuo": float(np.abs(entrena.iloc[-1] - real.values).mean()),
}
print({k: round(v, 2) for k, v in errores.items()})

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    _e = conteo[:"2024"]; _r = conteo["2025"].values
    assert abs(errores["promedio_estacional"] - np.abs(_e.groupby(_e.index.month).mean().values - _r).mean()) < 1e-9
    assert abs(errores["ingenuo"] - np.abs(_e.iloc[-1] - _r).mean()) < 1e-9
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'np.abs(entrena.groupby(entrena.index.month).mean().values - real.values).mean() y np.abs(entrena.iloc[-1] - real.values).mean().')
    print("   Detalle:", e)

## Para llevar

- Los sismos vienen **en racimos** (varianza/media ≈ 1,7): un terremoto trae decenas de réplicas.
  Eso basta para que un mes «parezca» más sísmico.
- Un test que supone independencia (chi-cuadrado sobre eventos) **se deja engañar** por los
  racimos; permutar meses completos no encuentra evidencia de estacionalidad.
- La validación lo confirma: el promedio estacional no le gana de forma apreciable al promedio
  plano. Lo honesto es pronosticar una **tasa** (≈ 22 sismos M≥4,5 al mes), no una «temporada».
- Saber decir **«no hay patrón»** también es un resultado. En el Estado, evita planificar en
  torno a un ciclo que no existe.

---

*Fin de la profundización de R2-10.*